In [27]:
# --- 0. INSTALL & IMPORTS ---
!pip install ultralytics -q
!pip install YOLOv8-Explainer
!pip install grad-cam -q
import os
import time
import glob
import shutil
import torch
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO
from IPython.display import display, clear_output
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, cohen_kappa_score, confusion_matrix,
    average_precision_score
)


In [28]:
# --- 1. SETTINGS & GPU CHECK ---
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print(f"Using Device: {DEVICE} ({torch.cuda.get_device_name(0) if DEVICE == 0 else 'CPU'})")

# File Paths
CSV_DIR      = "/kaggle/input/datasets/piageaguirre/modified-mura-dataset"
TRAIN_CSV    = os.path.join(CSV_DIR, "train_csv_path.csv")
VAL_CSV      = os.path.join(CSV_DIR, "valid_csv_path.csv")
TEST_CSV     = "/kaggle/input/datasets/piageaguirre/modified-mura-dataset/MURA-v1.1 (1)/MURA-v1.1/valid_image_paths.csv"
INPUT_BASE_DIR = "/kaggle/input/datasets/piageaguirre/modified-mura-dataset/MURA-v1.1 (1)"
TEMP_DATA_DIR  = "/kaggle/working/modified_dataset"

# ── Tunable knobs ──────────────────────────────────────────────────────────────
BODY_PARTS          = ['WRIST','HUMERUS','ELBOW']  # add/remove as needed
IMG_SIZE            = 512
MODEL_SIZE          = 'yolov8n-cls.pt'               # upgraded from nano
CONFIDENCE_THRESHOLD = 0.42  # lower than 0.5 to boost recall; tune 0.30–0.45
# ───────────────────────────────────────────────────────────────────────────────

all_run_metrics = []

Using Device: 0 (Tesla P100-PCIE-16GB)


In [29]:
# --- 2. DATA PREP FUNCTION ---
def prepare_data_for_part(part_name):
    if os.path.exists(TEMP_DATA_DIR):
        shutil.rmtree(TEMP_DATA_DIR)

    splits_to_process = {
        'train': TRAIN_CSV,
        'val':   VAL_CSV,
        'test':  TEST_CSV,
    }

    test_files_list = []

    for folder, csv_file in splits_to_process.items():
        if not os.path.exists(csv_file):
            continue
        df = pd.read_csv(csv_file, header=None, names=['rel_path'])
        part_df = df[df['rel_path'].str.contains(part_name, case=False)].copy()

        for rel_path in part_df['rel_path']:
            label     = 'positive' if 'positive' in rel_path.lower() else 'negative'
            src       = os.path.join(INPUT_BASE_DIR, rel_path)
            dst_folder = os.path.join(TEMP_DATA_DIR, folder, label)
            os.makedirs(dst_folder, exist_ok=True)
            dst_path  = os.path.join(dst_folder, rel_path.replace("/", "_"))

            if os.path.exists(src) and not os.path.exists(dst_path):
                os.symlink(src, dst_path)
                if folder == 'test':
                    test_files_list.append(dst_path)

    return test_files_list

In [30]:
# --- 3. LIVE PLOT CALLBACK ---
PLOT_DIR = "/kaggle/working/runs/mura_metric_plots"
os.makedirs(PLOT_DIR, exist_ok=True)

def on_train_epoch_end(trainer):
    """Refresh loss + accuracy plots after every epoch."""
    history_path = os.path.join(trainer.save_dir, 'results.csv')
    if not os.path.exists(history_path):
        return
    try:
        df = pd.read_csv(history_path)
        df.columns = [c.strip() for c in df.columns]
        epochs = df['epoch']

        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

        ax1.plot(epochs, df['train/loss'], label='Train Loss', marker='o', color='#1f77b4')
        ax1.plot(epochs, df['val/loss'],   label='Val Loss',   marker='o', color='#ff7f0e')
        ax1.set_title("LIVE: Loss History", fontweight='bold')
        ax1.legend(); ax1.grid(True, alpha=0.3)

        ax2.plot(epochs, df['metrics/accuracy_top1'], label='Val Accuracy', marker='o', color='#ff7f0e')
        ax2.set_title("LIVE: Accuracy History", fontweight='bold')
        ax2.set_ylim(0.4, 1.0)
        ax2.legend(loc='lower right'); ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()
    except Exception:
        pass  # Prevent crash if CSV is mid-write

In [31]:
# --- 4. PER-PART AUGMENTATION OVERRIDES ---
# Elbow laterality is clinically meaningful, so no horizontal flip there.
PART_OVERRIDES = {
    'WRIST':   {'fliplr': 0.5},
    'FOREARM': {'fliplr': 0.5},
    'ELBOW':   {'fliplr': 0.0},
}

In [32]:
# # --- 5. MAIN TRAINING LOOP ---
# for part in BODY_PARTS:
#     print(f"\n{'='*60}")
#     print(f" Processing: {part}")
#     print(f"{'='*60}")

#     # ── Data preparation ────────────────────────────────────────
#     test_files = prepare_data_for_part(part)
#     if not test_files:
#         print(f"  No test files found for {part}, skipping.")
#         continue

#     # ── Class balancing: oversample positives in train ───────────
#     train_pos = glob.glob(os.path.join(TEMP_DATA_DIR, 'train', 'positive', '*'))
#     train_neg = glob.glob(os.path.join(TEMP_DATA_DIR, 'train', 'negative', '*'))
#     n_pos, n_neg = len(train_pos), len(train_neg)
#     print(f"  Train split  →  Positive: {n_pos}  |  Negative: {n_neg}")

#     # if n_pos < n_neg:
#     #     copies_needed = n_neg - n_pos
#     #     for i in range(copies_needed):
#     #         src = train_pos[i % n_pos]
#     #         ext = os.path.splitext(src)[1]
#     #         dst = src.replace(ext, f'_dup{i}{ext}')
#     #         if not os.path.exists(dst):
#     #             shutil.copy2(src, dst)
#     #     print(f"  Oversampled {copies_needed} positive images to balance classes")

#     # ── Model init ───────────────────────────────────────────────
#     model = YOLO(MODEL_SIZE)
#     model.add_callback("on_train_epoch_end", on_train_epoch_end)

#     part_args = PART_OVERRIDES.get(part, {'fliplr': 0.5})

#     # ── Training ─────────────────────────────────────────────────
#     start_time = time.time()
#     train_results = model.train(
#         data=TEMP_DATA_DIR,
#         epochs=100,
#         imgsz=IMG_SIZE,
#         batch=32,               # Reduced from 32 for larger model
#         patience=7,            # Was 8 — give more room before early stop
#         lr0=0.00008,            # Was 0.00005 — much more active
#         lrf=0.01,
#         warmup_epochs=3,        # Gradual LR ramp-up
#         dropout=0.5,
#         weight_decay=0.005,    
#         label_smoothing=0.15,
#         optimizer='AdamW',
#         device=DEVICE,
#         # ── Augmentations ──
#         cos_lr=True,
#         degrees=15.0,
#         translate=0.1,
#         scale=0.4,
#         hsv_v=0.4,
#         erasing=0.3,            # NEW — random erasing for robustness
#         fliplr=part_args['fliplr'],
#         # hsv_h=0.0,              # Hue irrelevant for grayscale X-rays
#         # hsv_s=0.0,              # Saturation irrelevant for grayscale X-rays
#         # ── Misc ──
#         workers=3,
#         amp=True,
#         project='mura_test_eval',
#         name=f"Run2-{part.lower()}",
#         exist_ok=True,
#         verbose=False,
#     )
#     training_duration = time.time() - start_time
#     print(f"  Training finished in {training_duration/60:.1f} min")

#     # ── Final train accuracy ─────────────────────────────────────
#     print(f"  Calculating final training accuracy...")
#     best_model_path = os.path.join(train_results.save_dir, 'weights', 'best.pt')
#     # ── Save best model to dedicated output directory ────────────
#     MODEL_SAVE_DIR = "/kaggle/working/saved_models"
#     os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
#     saved_path = os.path.join(MODEL_SAVE_DIR, f"best_{part.lower()}.pt")
#     shutil.copy2(best_model_path, saved_path)
#     print(f"  ✅ Model saved → {saved_path}")
#     best_model = YOLO(best_model_path)
#     train_metrics  = best_model.val(split='train', verbose=False)
#     final_train_acc = train_metrics.top1

#     # ── Test metrics with custom confidence threshold ────────────
#     print(f"  Running inference on test set (threshold={CONFIDENCE_THRESHOLD})...")
#     # TO:
#     INFER_BATCH = 64
#     y_true, y_pred, y_probs = [], [], []
#     pos_idx = [k for k, v in best_model.names.items() if 'positive' in v.lower()][0]
    
#     for i in range(0, len(test_files), INFER_BATCH):
#         batch_files = test_files[i : i + INFER_BATCH]
#         batch_results = best_model.predict(
#             source=batch_files, imgsz=IMG_SIZE,
#             device=DEVICE, conf=0.0, verbose=False
#         )
#         for file_path, res in zip(batch_files, batch_results):
#             true_label    = 1 if "positive" in file_path.lower() else 0
#             prob_positive = res.probs.data[pos_idx].item()
#             pred_label    = 1 if prob_positive >= CONFIDENCE_THRESHOLD else 0
#             y_true.append(true_label)
#             y_pred.append(pred_label)
#             y_probs.append(prob_positive)

#     test_acc       = accuracy_score(y_true, y_pred)
#     test_precision = precision_score(y_true, y_pred, zero_division=0)
#     test_recall    = recall_score(y_true, y_pred, zero_division=0)
#     test_f1        = f1_score(y_true, y_pred, zero_division=0)
#     test_kappa     = cohen_kappa_score(y_true, y_pred)
#     test_ap        = average_precision_score(y_true, y_probs)
#     test_auc       = roc_auc_score(y_true, y_probs) if len(set(y_true)) > 1 else 0.5

#     cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
#     tn, fp, fn, tp = cm.ravel()
#     test_specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

#     # ── Final plots ──────────────────────────────────────────────
#     history_path = os.path.join(train_results.save_dir, 'results.csv')
#     history_df   = pd.read_csv(history_path)
#     history_df.columns = [c.strip() for c in history_df.columns]
#     epochs          = history_df['epoch']
#     val_acc_history = history_df['metrics/accuracy_top1']

#     clear_output(wait=True)
#     fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

#     ax1.plot(epochs, history_df['train/loss'], label='Train Loss', marker='o',
#              color='#1f77b4', linewidth=2)
#     ax1.plot(epochs, history_df['val/loss'],   label='Val Loss',   marker='o',
#              color='#ff7f0e', linewidth=2)
#     ax1.set_title(f"{part}: Loss History (Final)", fontweight='bold')
#     ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
#     ax1.legend(); ax1.grid(True, alpha=0.3)

#     ax2.plot(epochs, val_acc_history, label='Val Accuracy', marker='o',
#              color='#ff7f0e', linewidth=2)
#     ax2.scatter(epochs.iloc[-1], final_train_acc, color='#1f77b4', marker='s', s=150,
#                 label=f'Final Train Acc ({final_train_acc:.4f})', zorder=5)
#     ax2.scatter(epochs.iloc[-1], test_acc, color='green', marker='*', s=300,
#                 label=f'Final Test Acc ({test_acc:.4f})', zorder=6)
#     ax2.set_title(f"{part}: Accuracy Comparison", fontweight='bold')
#     ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy")
#     ax2.set_ylim(0.4, 1.0)
#     ax2.legend(loc='lower right'); ax2.grid(True, alpha=0.3)

#     plt.tight_layout()
#     plt.savefig(os.path.join(PLOT_DIR, f"{part.lower()}_metrics.png"), dpi=300)
#     plt.show()

#     # ── Append metrics ───────────────────────────────────────────
#     all_run_metrics.append({
#         "Body Part":              part,
#         "Confidence Threshold":   CONFIDENCE_THRESHOLD,
#         "Training Time (s)":      round(training_duration, 2),
#         "Training Time (m)":      round(training_duration / 60, 2),
#         "Accuracy":               test_acc,
#         "Precision":              test_precision,
#         "Recall (Sensitivity)":   test_recall,
#         "Specificity":            test_specificity,
#         "F1 Score":               test_f1,
#         "Cohen Kappa":            test_kappa,
#         "Average Precision (AP)": test_ap,
#         "AUC":                    test_auc,
#         "TP": int(tp), "TN": int(tn), "FP": int(fp), "FN": int(fn),
#         "Avg Train Accuracy":     final_train_acc,
#         "Avg Val Accuracy":       val_acc_history.iloc[-1],
#         "Avg Test Accuracy":      test_acc,
#     })

#     print(f"\n  Results for {part}:")
#     print(f"    Accuracy  : {test_acc:.4f}")
#     print(f"    Recall    : {test_recall:.4f}  (target ≥ 0.88)")
#     print(f"    Precision : {test_precision:.4f}")
#     print(f"    F1 Score  : {test_f1:.4f}")
#     print(f"    AUC       : {test_auc:.4f}")
#     print(f"    TP={int(tp)}  TN={int(tn)}  FP={int(fp)}  FN={int(fn)}")

In [33]:
# --- 6. SAVE & DISPLAY FINAL RESULTS ---
final_df = pd.DataFrame(all_run_metrics)
final_df.to_csv("/kaggle/working/mura_final_comprehensive.csv", index=False)

print("\n" + "="*60)
print(" EVALUATION COMPLETE")
print("="*60)
display(final_df)


 EVALUATION COMPLETE


""


In [34]:
# --- 7. THRESHOLD TUNING HELPER (optional) ---
# Run this cell AFTER training to find the best threshold for each body part
# without retraining. Looks at recall vs precision tradeoff.

import numpy as np
from sklearn.metrics import precision_recall_curve

# Re-run inference on a saved model — update the path below
# best_model = YOLO("/kaggle/working/runs/classify/mura_test_eval/Run2-wrist/weights/best.pt")

# Uncomment and run once you have test predictions (y_probs + y_true from cell 5)
# precisions, recalls, thresholds = precision_recall_curve(y_true, y_probs)

# plt.figure(figsize=(10, 6))
# plt.plot(thresholds, precisions[:-1], label='Precision', color='blue')
# plt.plot(thresholds, recalls[:-1],    label='Recall',    color='red')
# plt.axvline(x=CONFIDENCE_THRESHOLD, color='green', linestyle='--',
#             label=f'Current threshold ({CONFIDENCE_THRESHOLD})')
# plt.xlabel('Threshold')
# plt.title('Precision vs Recall at different thresholds')
# plt.legend()
# plt.grid(True, alpha=0.3)
# plt.tight_layout()
# plt.show()

# # Find threshold where recall >= 0.88
# target_recall = 0.88
# valid = np.where(recalls[:-1] >= target_recall)[0]
# if len(valid):
#     best_idx = valid[np.argmax(precisions[valid])]
#     print(f"Best threshold for recall >= {target_recall}: {thresholds[best_idx]:.3f}")
#     print(f"  Precision at that point: {precisions[best_idx]:.4f}")
#     print(f"  Recall    at that point: {recalls[best_idx]:.4f}")

print("Threshold tuning cell ready — uncomment to use after training.")

Threshold tuning cell ready — uncomment to use after training.


In [40]:
import torch
from ultralytics import YOLO

# 1. Load your best model
model = YOLO('/kaggle/working/saved_models/best_humerus.pt')
pt_model = model.model  # Get the underlying PyTorch model

# 2. Modify the model to output both prediction and feature maps
class GradCAMModel(torch.nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.backbone = base_model.model[:-1] # All layers except the final head
        self.head = base_model.model[-1]      # The classification head

    def forward(self, x):
        features = self.backbone(x)           # Shape: [1, 1280, 7, 7] (approx)
        preds = self.head(features)           # Shape: [1, 2]
        return preds, features

# 3. Export to TFLite
dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
wrapper = GradCAMModel(pt_model)
torch.onnx.export(wrapper, dummy_input, "multi_output.onnx", opset_version=11)

# Now convert ONNX to TFLite (using a tool like onnx2tf or similar)
# For now, export standard TFLite and we will use the weights trick in Flutter
model.export(format='tflite', imgsz=IMG_SIZE)

Ultralytics 8.4.41 🚀 Python-3.12.12 torch-2.8.0+cu126 CPU (Intel Xeon CPU @ 2.00GHz)
YOLOv8n-cls summary (fused): 30 layers, 1,437,442 parameters, 0 gradients, 3.3 GFLOPs

PyTorch: starting from '/kaggle/working/saved_models/best_humerus.pt' with input shape (1, 3, 512, 512) BCHW and output shape(s) (1, 2) (2.8 MB)

TensorFlow SavedModel: starting export with tensorflow 2.19.0...

ONNX: starting export with onnx 1.20.1 opset 22...
ONNX: slimming with onnxslim 0.1.91...
ONNX: export success ✅ 0.4s, saved as '/kaggle/working/saved_models/best_humerus.onnx' (5.5 MB)
TensorFlow SavedModel: starting TFLite export with onnx2tf 1.28.8...
Saved artifact at '/kaggle/working/saved_models/best_humerus_saved_model'. The following endpoints are available:

* Endpoint 'serving_default'
  inputs_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 512, 512, 3), dtype=tf.float32, name='images')
Output Type:
  TensorSpec(shape=(1, 2), dtype=tf.float32, name=None)
Captures:
  132664831627280: TensorSpec(shape=(4, 

I0000 00:00:1776933488.342604      55 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
I0000 00:00:1776933488.342847      55 single_machine.cc:374] Starting new session
I0000 00:00:1776933488.343749      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
W0000 00:00:1776933488.624435      55 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1776933488.624468      55 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
I0000 00:00:1776933488.904221      55 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
I0000 00:00:1776933488.904433      55 single_machine.cc:374] Starting new session
I0000 00:00:1776933488.905363      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0

TensorFlow SavedModel: export success ✅ 4.3s, saved as '/kaggle/working/saved_models/best_humerus_saved_model' (13.8 MB)

TensorFlow Lite: starting export with tensorflow 2.19.0...
TensorFlow Lite: export success ✅ 0.0s, saved as '/kaggle/working/saved_models/best_humerus_saved_model/best_humerus_float32.tflite' (5.5 MB)

Export complete (4.5s)
Results saved to /kaggle/working/saved_models
Predict:         yolo predict task=classify model=/kaggle/working/saved_models/best_humerus_saved_model/best_humerus_float32.tflite imgsz=512 
Validate:        yolo val task=classify model=/kaggle/working/saved_models/best_humerus_saved_model/best_humerus_float32.tflite imgsz=512 data=/kaggle/working/modified_dataset  
Visualize:       https://netron.app


W0000 00:00:1776933489.156160      55 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1776933489.156193      55 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.


'/kaggle/working/saved_models/best_humerus_saved_model/best_humerus_float32.tflite'

In [36]:
# --- 8. ZIP OUTPUTS ---
!zip -r /kaggle/working/mura_results.zip /kaggle/working/runs /kaggle/working/saved_models /kaggle/working/mura_final_comprehensive.csv


updating: kaggle/working/runs/ (stored 0%)
updating: kaggle/working/runs/mura_metric_plots/ (stored 0%)
updating: kaggle/working/runs/mura_metric_plots/wrist_metrics.png (deflated 17%)
updating: kaggle/working/runs/mura_metric_plots/forearm_metrics.png (deflated 16%)
updating: kaggle/working/runs/mura_metric_plots/elbow_metrics.png (deflated 17%)
updating: kaggle/working/runs/classify/ (stored 0%)
updating: kaggle/working/runs/classify/mura_test_eval/ (stored 0%)
updating: kaggle/working/runs/classify/mura_test_eval/Run2-wrist/ (stored 0%)
updating: kaggle/working/runs/classify/mura_test_eval/Run2-wrist/results.png (deflated 10%)
updating: kaggle/working/runs/classify/mura_test_eval/Run2-wrist/train_batch2.jpg (deflated 23%)
updating: kaggle/working/runs/classify/mura_test_eval/Run2-wrist/val_batch0_labels.jpg (deflated 16%)
updating: kaggle/working/runs/classify/mura_test_eval/Run2-wrist/weights/ (stored 0%)
updating: kaggle/working/runs/classify/mura_test_eval/Run2-wrist/weights/last

In [37]:
# !rm -rf /kaggle/working/*